# 🧬 maniFasta on Google Colab

[![Open maniFasta in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KauffmanLab/maniFasta/blob/main/maniFasta_on_GoogleColab.ipynb)

### Build a custom protein FASTA database without editing registry or config files.

**Compatibility and provenance:** this notebook loads the repository version shown in Step 1 by default. The notebook and scripts are maintained together in the same repository, and every completed build records the exact maniFasta Git tag and commit used.

**What this notebook does:** it walks you through a few questions, lets you choose curated sources and/or upload your own files, checks everything with maniFasta itself, runs the build, and gives you one complete results ZIP.

| ① Load maniFasta | ② Choose sources | ③ Optional uploads | ④ Name + validate | ⑤ Build | ⑥ Download |
|---|---|---|---|---|---|
| Load maniFasta | Pick curated inputs | Add proteomes, accessions, or FASTA | Review the recipe | Run maniFasta | Save the whole run |

> **Good news:** you do **not** need to edit code, the source registry, or the config file by hand. The notebook creates the Colab setup for you and leaves the maniFasta scripts unchanged.

> **Independent sessions:** each person who opens the notebook in Colab receives a separate temporary runtime. Files and results in `/content` are not shared with other users and disappear when that runtime is reset.

---

## Choose your path

| Your situation | Best route through this notebook |
|---|---|
| 📦 **Only curated/prepackaged sources** | Make choices in **Question 2**, then skip uploads. |
| 📤 **Only my own files** | Leave **Question 2** empty, then use **Question 3**. |
| 🧩 **Curated sources plus my own files** | Choose curated sources in **Question 2**, then add files in **Question 3**. |
| 🔐 **Private GitHub or repository ZIP** | Use a hidden GitHub token or upload a repository ZIP in **Question 1**. |

<div style="background:#f0fdf4; border:1px solid #bbf7d0; border-radius:14px; padding:15px 18px; margin:16px 0;">
  <b>Main rule:</b> run the notebook from top to bottom. Each section tells you whether it can be skipped.
</div>


<div style="background:#eef6ff; border-left:7px solid #2563eb; border-radius:16px; padding:18px 20px; margin:18px 0;">
  <div style="font-size:23px; font-weight:900; color:#0f172a;">Question 1 — Where should the notebook get maniFasta?</div>
  <p style="color:#334155; line-height:1.6; margin-bottom:0;">Choose one loading method. The next cell shows only the fields that apply and checks for the current registry and planner files.</p>
</div>

<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(235px,1fr)); gap:12px; margin-bottom:10px;">
  <div style="background:#f0fdf4; border:1px solid #bbf7d0; border-radius:14px; padding:14px;"><b>🌐 Public GitHub repo</b><br><span style="color:#334155;">No token needed.</span></div>
  <div style="background:#fff7ed; border:1px solid #fed7aa; border-radius:14px; padding:14px;"><b>🔐 Private GitHub repo</b><br><span style="color:#334155;">Enter a hidden personal access token. It is used only for this clone.</span></div>
  <div style="background:#f8fafc; border:1px solid #cbd5e1; border-radius:14px; padding:14px;"><b>📦 Repository ZIP</b><br><span style="color:#334155;">Useful for a local version or repository snapshot.</span></div>
</div>

<div style="background:#fefce8; border-left:7px solid #ca8a04; border-radius:14px; padding:14px 17px; margin:12px 0;">
  <b>Private repository?</b> The token must be able to read this repository. The notebook never places the token in the clone URL, terminal output, repository files, or results ZIP.
</div>


In [ ]:
#@title Step 1. Prepare maniFasta { display-mode: "form" }
#@markdown Click play, choose GitHub or a repository ZIP, then press **Prepare maniFasta**.
#@markdown This loads maniFasta and checks the command-line tools required for the build.

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML, clear_output
import csv, gzip, hashlib, html, io, json, os, re, shutil, subprocess, sys, tempfile, zipfile
try:
    import ipywidgets as widgets
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=True)
    import ipywidgets as widgets

CONTENT_DIR = Path('/content')
REPO_DIR = CONTENT_DIR / 'maniFasta'
RUN_ROOT = CONTENT_DIR / 'maniFasta_runs'
UPLOAD_ROOT = REPO_DIR / '00.setup' / 'user_uploads'
TEMPLATE_DIR = CONTENT_DIR / 'maniFasta_input_templates'

SOURCE_ROWS = []
SOURCE_CONTROLS = {}
CUSTOM_SOURCE_CARDS = []
LAST_RUN_DIR = None
LAST_RUN_ZIP = None
MANIFASTA_GIT_COMMIT = None
COLAB_PHASE_RECORDS = []

def _record_colab_phase(phase, started, finished=None, status='completed', detail=''):
    finished = finished or datetime.now(timezone.utc)
    COLAB_PHASE_RECORDS.append({
        'phase': str(phase),
        'started_utc': started.isoformat().replace('+00:00', 'Z'),
        'finished_utc': finished.isoformat().replace('+00:00', 'Z'),
        'elapsed_seconds': max(0.0, (finished - started).total_seconds()),
        'status': str(status),
        'detail': str(detail).replace('\t', ' ').replace('\n', '; '),
    })

def _write_colab_phase_timings(run_dir):
    path = Path(run_dir) / 'COLAB_PHASE_TIMINGS.tsv'
    columns = ['phase', 'started_utc', 'finished_utc', 'elapsed_seconds', 'status', 'detail']
    with path.open('w', encoding='utf-8', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, delimiter='\t', lineterminator='\n')
        writer.writeheader()
        writer.writerows(COLAB_PHASE_RECORDS)
    return path

display(HTML(r"""
<style>
.mf-section {background:#f7fbfe;border:1px solid #d8e8f2;border-radius:16px;padding:18px 20px;margin:10px 0 14px 0}
.mf-good {background:#edf8f0;border:1px solid #c5e4cc;border-radius:12px;padding:12px 15px;color:#315b3b}
.mf-note {background:#fff7e7;border:1px solid #efdcad;border-radius:12px;padding:12px 15px;color:#66582f}
.mf-error {background:#fff0f0;border:1px solid #efcaca;border-radius:12px;padding:12px 15px;color:#7c3535}
.mf-title {font-size:19px;font-weight:800;color:#244f68;margin:2px 0 9px}
.mf-muted {color:#637581;font-size:13px;line-height:1.45}
.widget-label {font-weight:650 !important}
</style>
"""))

repo_mode = widgets.ToggleButtons(
    options=[
        ('Public GitHub', 'github_public'),
        ('Private GitHub + token', 'github_private'),
        ('Upload a repository ZIP', 'zip'),
    ],
    value='github_public',
    description='Repository:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='720px')
)
repo_url = widgets.Text(
    value='https://github.com/KauffmanLab/maniFasta.git',
    description='GitHub URL:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='720px')
)
repo_branch = widgets.Text(
    value='main',
    placeholder='Branch or tag, e.g. main',
    description='Branch or tag:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='720px')
)
repo_token = widgets.Password(
    value='',
    placeholder='Hidden GitHub personal access token',
    description='GitHub token:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='720px')
)
repo_token_help = widgets.HTML(
    '<div class="mf-muted" style="margin:0 0 8px 128px;max-width:820px">'
    'Used only for a private HTTPS clone, then cleared from the form. '
    'The token is never added to the repository URL or written to disk.</div>'
)
repo_zip = widgets.FileUpload(accept='.zip', multiple=False, description='Choose repository ZIP')
prepare_button = widgets.Button(
    description='Prepare maniFasta', icon='check', button_style='success',
    layout=widgets.Layout(width='220px', height='40px')
)
prepare_output = widgets.Output()


def _uploaded_items(upload_widget):
    value = upload_widget.value
    if isinstance(value, dict):
        for name, meta in value.items():
            content = meta.get('content', b'')
            yield name, bytes(content)
    else:
        for meta in value:
            name = meta.get('name', 'upload.bin')
            content = meta.get('content', b'')
            yield name, bytes(content)


def _safe_extract_zip(zip_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if target != base and base not in target.parents:
                raise ValueError(f'Unsafe path in ZIP: {member.filename}')
        zf.extractall(destination)


def _find_repo_root(search_root: Path) -> Path:
    candidates = []
    for p in [search_root, *search_root.rglob('*')]:
        if p.is_dir() and (p / '00.setup').is_dir() and (p / '01.scripts').is_dir():
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError('Could not find a folder containing both 00.setup and 01.scripts.')
    return min(candidates, key=lambda p: len(p.parts))


def _ensure_datasets_cli() -> str:
    found = shutil.which('datasets')
    if found:
        return found
    destination = Path('/usr/local/bin/datasets')
    url = 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
    subprocess.run(
        ['curl', '-fL', '--retry', '4', '--retry-delay', '2', '-o', str(destination), url],
        check=True,
    )
    destination.chmod(0o755)
    subprocess.run([str(destination), 'version'], check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return str(destination)


def _clone_github_repo(*, url: str, destination: Path, branch: str = '', token: str = ''):
    url = (url or '').strip()
    if not url:
        raise ValueError('Enter the GitHub repository URL.')

    cmd = ['git', 'clone', '--depth', '1', '--single-branch', '--filter=blob:none']
    if branch.strip():
        cmd += ['--branch', branch.strip()]
    cmd += [url, str(destination)]

    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    askpass_dir = None
    if token:
        askpass_dir = Path(tempfile.mkdtemp(prefix='manifasta_git_auth_'))
        askpass = askpass_dir / 'askpass.sh'
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "${MF_GITHUB_USERNAME:-x-access-token}" ;;\n'
            '  *)          printf "%s\\n" "${MF_GITHUB_TOKEN}" ;;\n'
            'esac\n',
            encoding='utf-8',
        )
        askpass.chmod(0o700)
        env['GIT_ASKPASS'] = str(askpass)
        env['MF_GITHUB_USERNAME'] = 'x-access-token'
        env['MF_GITHUB_TOKEN'] = token

    try:
        result = subprocess.run(
            cmd, env=env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
    finally:
        env.pop('MF_GITHUB_TOKEN', None)
        if askpass_dir is not None:
            shutil.rmtree(askpass_dir, ignore_errors=True)

    if result.returncode != 0:
        detail = (result.stdout or '').strip()
        if token:
            detail = detail.replace(token, '[REDACTED]')
        hint = (
            'For a private repository, choose “Private GitHub + token” and use a token '
            'that has read access to this repository. Also confirm the URL and branch/tag.'
        )
        raise RuntimeError(
            f'Git clone failed (status {result.returncode}).\n{detail[-3500:]}\n\n{hint}'
        )


def _prepare_repo(_button=None):
    global REPO_DIR, UPLOAD_ROOT, MANIFASTA_GIT_COMMIT
    phase_started = datetime.now(timezone.utc)
    phase_status = 'failed'
    phase_detail = ''
    staging_root = None
    token = ''
    mode = repo_mode.value
    with prepare_output:
        clear_output()
        try:
            token = repo_token.value.strip() if mode == 'github_private' else ''
            zip_items = None

            # Validate the requested source before replacing a working repository.
            if mode == 'github_private' and not token:
                raise ValueError('Enter a GitHub token for the private-repository option.')
            if mode == 'zip':
                zip_items = list(_uploaded_items(repo_zip))
                if not zip_items:
                    raise ValueError('Choose a repository ZIP first.')

            display(HTML(
                '<div class="mf-note"><b>Preparing the workspace…</b> '
                'The existing Colab copy will be replaced only after the new copy passes its checks.</div>'
            ))

            staging_root = Path(tempfile.mkdtemp(prefix='manifasta_prepare_'))
            staged_repo = staging_root / 'maniFasta'

            if mode in {'github_public', 'github_private'}:
                _clone_github_repo(
                    url=repo_url.value,
                    destination=staged_repo,
                    branch=repo_branch.value,
                    token=token,
                )
            else:
                zip_path = staging_root / Path(zip_items[0][0]).name
                zip_path.write_bytes(zip_items[0][1])
                unpacked = staging_root / 'unpacked'
                _safe_extract_zip(zip_path, unpacked)
                root = _find_repo_root(unpacked)
                shutil.copytree(root, staged_repo)

            required = [
                staged_repo / '00.setup' / '000.maniFasta.config',
                staged_repo / '00.setup' / '000.maniFasta.source_registry.tsv',
                staged_repo / '01.scripts' / 'runBuild_db.sh',
                staged_repo / '01.scripts' / 'source_planner.py',
            ]
            missing = [str(p.relative_to(staged_repo)) for p in required if not p.is_file()]
            if missing:
                raise FileNotFoundError(
                    'Repository is missing required current files:\n' + '\n'.join(missing)
                )

            datasets_path = _ensure_datasets_cli()

            if REPO_DIR.exists():
                shutil.rmtree(REPO_DIR)
            shutil.move(str(staged_repo), str(REPO_DIR))

            RUN_ROOT.mkdir(parents=True, exist_ok=True)
            UPLOAD_ROOT = REPO_DIR / '00.setup' / 'user_uploads'
            UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)

            commit_result = subprocess.run(
                ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
                text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            )
            MANIFASTA_GIT_COMMIT = (
                commit_result.stdout.strip()
                if commit_result.returncode == 0 and commit_result.stdout.strip()
                else 'Unavailable (repository ZIP or non-Git source)'
            )

            phase_status = 'completed'
            display(HTML(
                '<div class="mf-good"><b>✓ maniFasta is ready.</b><br>'
                f'Repository: <code>{REPO_DIR}</code><br>'
                f'Exact Git commit: <code>{html.escape(MANIFASTA_GIT_COMMIT)}</code><br>'
                f'NCBI datasets: <code>{datasets_path}</code><br>'
                f'Repository access: <code>{mode.replace("_", " ")}</code></div>'
            ))
        except Exception as exc:
            phase_detail = str(exc)
            detail = html.escape(str(exc))
            display(HTML(
                f'<div class="mf-error"><b>Setup did not finish:</b>'
                f'<br><pre style="white-space:pre-wrap">{detail}</pre></div>'
            ))
            return
        finally:
            # Do not leave a private token in the visible widget after any clone attempt.
            if mode == 'github_private':
                repo_token.value = ''
            if staging_root is not None:
                shutil.rmtree(staging_root, ignore_errors=True)
            _record_colab_phase('prepare_repository', phase_started, status=phase_status, detail=phase_detail)

prepare_button.on_click(_prepare_repo)

def _toggle_repo_fields(change=None):
    is_git = repo_mode.value in {'github_public', 'github_private'}
    is_private = repo_mode.value == 'github_private'
    repo_url.layout.display = '' if is_git else 'none'
    repo_branch.layout.display = '' if is_git else 'none'
    repo_token.layout.display = '' if is_private else 'none'
    repo_token_help.layout.display = '' if is_private else 'none'
    repo_zip.layout.display = 'none' if is_git else ''
repo_mode.observe(_toggle_repo_fields, names='value')
_toggle_repo_fields()

display(HTML('<div class="mf-section"><div class="mf-title">Repository source</div><div class="mf-muted">The default repository version is shown above. The exact checked-out Git commit is shown after preparation and recorded with completed runs. GitHub clones start clean; use the ZIP option to run from a custom repository copy.</div></div>'))
display(repo_mode, repo_url, repo_branch, repo_token, repo_token_help, repo_zip, prepare_button, prepare_output)


<div style="background:#eef6ff; border-left:7px solid #2563eb; border-radius:16px; padding:18px 20px; margin:18px 0;">
  <div style="font-size:23px; font-weight:900; color:#0f172a;">Question 2 — Which prepackaged sources should be included?</div>
  <p style="color:#334155; line-height:1.6;">The chooser begins empty. Select only the sources appropriate for your study, or load the four-source example to try every maniFasta module.</p>
</div>

<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(245px,1fr)); gap:12px; margin:10px 0 16px 0;">
  <div style="background:#f0fdf4; border:1px solid #bbf7d0; border-radius:14px; padding:14px;"><b>🧪 Example build</b><br><span style="color:#334155;">Load one source per module to try the workflow. This is not a scientific recommendation.</span></div>
  <div style="background:#fff7ed; border:1px solid #fed7aa; border-radius:14px; padding:14px;"><b>📤 My own files only</b><br><span style="color:#334155;">Leave all prepackaged choices empty, then add files in Question 3.</span></div>
  <div style="background:#f8fafc; border:1px solid #cbd5e1; border-radius:14px; padding:14px;"><b>🧩 Mix and match</b><br><span style="color:#334155;">Choose any curated sources you need and add your own afterward.</span></div>
</div>

<div style="background:#fff7ed; border-left:7px solid #ea580c; border-radius:14px; padding:15px 18px; margin:12px 0;">
  <b>Human and HOMD are special:</b> each menu allows <b>none or one</b> flavor only. Other curated sources can be combined freely.
</div>


In [ ]:
#@title Step 2. Choose prepackaged sources { display-mode: "form" }
#@markdown Click play to open the source chooser.
#@markdown Human and HOMD are each a none-or-one menu; all other curated sources are independent checkboxes.

from collections import defaultdict

registry_path = REPO_DIR / '00.setup' / '000.maniFasta.source_registry.tsv'
if not registry_path.is_file():
    raise RuntimeError('Run Step 1 and click “Prepare maniFasta” first.')


def read_registry_rows(path: Path):
    with path.open(newline='', encoding='utf-8') as fh:
        lines = [ln for ln in fh if ln.strip() and not ln.startswith('#')]
    reader = csv.DictReader(lines, delimiter='\t')
    return [dict(r) for r in reader]


def parse_options(raw: str):
    out = {}
    for item in (raw or '').split(';'):
        item = item.strip()
        if not item:
            continue
        if '=' in item:
            k, v = item.split('=', 1)
            out[k.strip()] = v.strip()
    return out


def truthy(value):
    return str(value or '').strip().lower() in {'true', 't', 'yes', 'y', '1', 'on'}


def row_description(row):
    notes = (row.get('notes') or '').strip()
    version = (row.get('version') or '').strip()
    bits = []
    if version:
        bits.append(f'v. {version}')
    if notes:
        bits.append(notes)
    return ' · '.join(bits) or (row.get('source_label') or row.get('source_id') or 'source')


SOURCE_ROWS = read_registry_rows(registry_path)
EXAMPLE_SOURCE_IDS = {'VIRUSES_HUMAN', 'VIRUSES_HERVS', 'MICROEUK_TTENAX', 'HUMAN_CI'}
human_rows = [r for r in SOURCE_ROWS if (r.get('collection') or '').strip().lower() == 'human_uniprot']
homd_rows = [r for r in SOURCE_ROWS if (r.get('collection') or '').strip().lower() == 'homd']
other_rows = [r for r in SOURCE_ROWS if r not in human_rows and r not in homd_rows]


def human_option(row):
    protein_set = parse_options(row.get('options', '')).get('protein_set', 'canonical')
    pretty = {
        'canonical': 'Canonical reviewed proteins (~20k)',
        'canonical_isoform': 'Canonical + reviewed isoforms (~42k)',
        'canonical_isoform_trembl': 'Canonical + isoforms + TrEMBL (~250k+)',
    }.get(protein_set, protein_set.replace('_', ' ').title())
    return f'{pretty}  [{row.get("source_label")}]'


def homd_option(row):
    opts = parse_options(row.get('options', ''))
    rank = opts.get('rank', '').strip().lower()
    pretty = {
        '': 'All oral HOMD assemblies (no dereplication)',
        'species': 'One representative genome per oral species',
        'genus': 'One representative genome per oral genus',
        'family': 'One representative genome per oral family',
    }.get(rank, f'One representative per {rank}')
    version = opts.get('genomic_refseq_version') or row.get('version') or 'current release'
    return f'{pretty} · {version}  [{row.get("source_label")}]'


human_choice = widgets.Dropdown(
    options=[('No human proteins', '')] + [(human_option(r), r['source_id']) for r in human_rows],
    value='',
    description='Human:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='850px')
)
homd_choice = widgets.Dropdown(
    options=[('No HOMD proteins', '')] + [(homd_option(r), r['source_id']) for r in homd_rows],
    value='',
    description='HOMD:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='850px')
)

other_checks = {}
grouped = defaultdict(list)
for row in other_rows:
    grouped[(row.get('source_group') or 'other').strip()].append(row)

source_summary = widgets.Output()


def _refresh_source_summary(change=None):
    with source_summary:
        clear_output()
        selected = []
        if human_choice.value:
            selected.append(human_choice.value)
        if homd_choice.value:
            selected.append(homd_choice.value)
        selected += [sid for sid, cb in other_checks.items() if cb.value]
        if selected:
            display(HTML(f'<div class="mf-good"><b>✓ {len(selected)} prepackaged source(s) selected.</b> You may still add your own sources below.</div>'))
        else:
            display(HTML('<div class="mf-note"><b>No prepackaged sources selected.</b> That is fine when you will add at least one source of your own.</div>'))


example_recipe_button = widgets.Button(
    description='Load example set — one per module', icon='flask', button_style='success',
    layout=widgets.Layout(width='330px', height='40px')
)
clear_recipe_button = widgets.Button(
    description='Start with no prepackaged sources', icon='eraser', button_style='warning',
    layout=widgets.Layout(width='295px', height='40px')
)
recipe_output = widgets.Output()

def _clear_recipe(_button=None):
    human_choice.value = ''
    homd_choice.value = ''
    for cb in other_checks.values():
        cb.value = False
    with recipe_output:
        clear_output()
        display(HTML(
            '<div class="mf-note"><b>Prepackaged sources cleared.</b> '
            'Choose sources below or add at least one custom source in Question 3.</div>'
        ))
    _refresh_source_summary()


def _load_example_recipe(_button=None):
    available_ids = {r.get('source_id') for r in SOURCE_ROWS}
    missing = sorted(EXAMPLE_SOURCE_IDS - available_ids)
    with recipe_output:
        clear_output()
        if missing:
            display(HTML(
                '<div class="mf-error"><b>The complete example set is not available in this repository.</b><br>'
                f'Missing source IDs: <code>{html.escape(", ".join(missing))}</code></div>'
            ))
            return

    human_choice.value = 'HUMAN_CI'
    homd_choice.value = ''
    for sid, cb in other_checks.items():
        cb.value = sid in EXAMPLE_SOURCE_IDS
    with recipe_output:
        clear_output()
        display(HTML(
            '<div class="mf-good"><b>✓ Example set loaded.</b> One source is selected for each '
            'maniFasta module. This demonstrates the workflow; it is <b>not</b> a recommended '
            'database composition.</div>'
        ))
    _refresh_source_summary()


example_recipe_button.on_click(_load_example_recipe)
clear_recipe_button.on_click(_clear_recipe)

cards = []
for group in sorted(grouped):
    children = [widgets.HTML(f'<div class="mf-title" style="font-size:16px">{group.replace("_", " ").title()}</div>')]
    for row in grouped[group]:
        sid = row['source_id']
        cb = widgets.Checkbox(
            value=False,
            description=f'{row.get("source_label")}: {row_description(row)}',
            indent=False,
            layout=widgets.Layout(width='100%'),
            style={'description_width': 'initial'},
        )
        cb.observe(_refresh_source_summary, names='value')
        other_checks[sid] = cb
        children.append(cb)
    cards.append(widgets.VBox(children, layout=widgets.Layout(
        border='1px solid #dce8ef', padding='13px 15px', margin='0 0 10px 0', width='100%'
    )))

human_choice.observe(_refresh_source_summary, names='value')
homd_choice.observe(_refresh_source_summary, names='value')
SOURCE_CONTROLS = {
    'human': human_choice,
    'homd': homd_choice,
    'other': other_checks,
}

display(HTML('''
<div class="mf-section">
  <div class="mf-title">Choose a starting point</div>
  <div class="mf-muted">The chooser starts empty. Select sources individually, or load an example that exercises all four maniFasta modules:</div>
  <div style="margin-top:10px;line-height:1.65">
    <code>VIRUSES_HUMAN</code> (Module I) · <code>VIRUSES_HERVS</code> (Module II) ·
    <code>MICROEUK_TTENAX</code> (Module III) · <code>HUMAN_CI</code> (Module IV)
  </div>
  <div class="mf-note" style="margin-top:12px"><b>Example only:</b> this combination demonstrates the workflow and is not a recommended database composition. Choose sources appropriate for your study.</div>
</div>
'''))
display(widgets.HBox([example_recipe_button, clear_recipe_button]), recipe_output)
display(HTML('<div class="mf-section"><div class="mf-title">Human and HOMD — choose none or one</div><div class="mf-muted">The menus make mutually exclusive choices automatic.</div></div>'))
display(human_choice, homd_choice)
display(HTML('<div class="mf-section"><div class="mf-title">Other curated and supported sources</div><div class="mf-muted">Check any combination. No sources are selected automatically.</div></div>'))
display(*cards, source_summary)
_refresh_source_summary()


<div style="background:#fff7ed; border-left:7px solid #ea580c; border-radius:16px; padding:18px 20px; margin:18px 0;">
  <div style="font-size:23px; font-weight:900; color:#0f172a;">Optional Question 3 — Would you like to add your own files?</div>
  <p style="color:#334155; line-height:1.6; margin-bottom:0;">Upload files first, then click the button matching what you have. You can add more than one custom source.</p>
</div>

<div style="background:#f0fdf4; border:1px solid #bbf7d0; border-radius:14px; padding:16px 18px; margin:12px 0;">
  <b>✅ Using only prepackaged sources?</b><br><span style="color:#334155;">Skip this section and go directly to <b>Question 4 — Name and validate the run</b>.</span>
</div>

## What kind of input file do you have?

| File you have | Choose | What maniFasta does |
|---|---|---|
| **Species, taxon IDs, genome accessions, or proteome IDs** | **Module I — proteomes** | Retrieves whole proteomes from NCBI or UniProt. |
| **Individual protein accessions** | **Module II — proteins** | Retrieves proteins from NCBI, UniProt, UniParc, or PDB. |
| **A ready-made protein FASTA** | **Module III — FASTA** | Uses your sequences directly; metadata is optional. |

<div style="background:#eef6ff; border:1px solid #bfdbfe; border-radius:14px; padding:14px 16px; margin:12px 0;">
  <b>ZIPs and compressed files are welcome.</b> The notebook safely unpacks ZIPs and decompresses <code>.gz</code> files before showing them in the source-card menus.
</div>


In [ ]:
#@title Step 3. Optional — upload and describe your own sources { display-mode: "form" }
#@markdown Skip this cell when you are using only prepackaged sources.
#@markdown Otherwise, upload files first and then add one source card for each dataset.

if not REPO_DIR.is_dir():
    raise RuntimeError('Run Step 1 first.')
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)

(TEMPLATE_DIR / 'module_I_proteome_list.tsv').write_text(
    'species\taccession\ttaxonID\tfetch_source\n'
    'Porphyromonas gingivalis\tGCF_000007585.1\t837\tncbi\n'
    'Escherichia coli\tUP000000625\t562\tuniprot\n', encoding='utf-8')
(TEMPLATE_DIR / 'module_II_protein_accessions.tsv').write_text(
    'accession\tfetch_source\tnote\n'
    'NP_000537.3\tncbi\texample NCBI protein accession\n'
    'P04637\tuniprot\texample UniProt accession\n', encoding='utf-8')
(TEMPLATE_DIR / 'module_III_metadata.tsv').write_text(
    'protein_id\tsource\tncbi_taxid\tgenus\tspecies\tstrain_or_gene\tdescription\n'
    'protein_001\tMY_SOURCE\t837\tPorphyromonas\tgingivalis\t\tExample protein\n', encoding='utf-8')

upload_widget = widgets.FileUpload(
    accept='.zip,.tsv,.txt,.csv,.fasta,.fa,.faa,.fas,.fsa,.gz',
    multiple=True,
    description='Choose input files'
)
save_uploads_button = widgets.Button(
    description='Save uploaded files', icon='upload', button_style='info',
    layout=widgets.Layout(width='220px')
)
add_proteome_button = widgets.Button(
    description='I have a proteome/species list', icon='list', button_style='info',
    layout=widgets.Layout(width='260px', height='40px')
)
add_protein_button = widgets.Button(
    description='I have protein accessions', icon='list-ol', button_style='info',
    layout=widgets.Layout(width='235px', height='40px')
)
add_fasta_button = widgets.Button(
    description='I have a protein FASTA', icon='file', button_style='success',
    layout=widgets.Layout(width='225px', height='40px')
)
upload_output = widgets.Output()
custom_cards_box = widgets.VBox([])
CUSTOM_SOURCE_CARDS = []


def available_upload_files():
    if not UPLOAD_ROOT.exists():
        return []
    return sorted(
        str(p.relative_to(UPLOAD_ROOT)) for p in UPLOAD_ROOT.rglob('*')
        if p.is_file() and '__MACOSX' not in p.parts and not any(part.startswith('.') for part in p.relative_to(UPLOAD_ROOT).parts)
    )


def refresh_card_files():
    options = [('', '')] + [(f, f) for f in available_upload_files()]
    for card in CUSTOM_SOURCE_CARDS:
        for key in ('primary_file', 'metadata_file'):
            widget = card[key]
            previous = widget.value
            widget.options = options
            if previous in [v for _, v in options]:
                widget.value = previous


def save_uploads(_button=None):
    with upload_output:
        clear_output()
        try:
            items = list(_uploaded_items(upload_widget))
            if not items:
                raise ValueError('Choose one or more files first.')
            saved = []
            for name, content in items:
                safe_name = Path(name).name
                target = UPLOAD_ROOT / safe_name
                target.write_bytes(content)
                if target.suffix.lower() == '.zip':
                    extract_to = UPLOAD_ROOT / target.stem
                    _safe_extract_zip(target, extract_to)
                    saved.extend(str(p.relative_to(UPLOAD_ROOT)) for p in extract_to.rglob('*') if p.is_file())
                elif target.suffix.lower() == '.gz':
                    uncompressed = target.with_suffix('')
                    with gzip.open(target, 'rb') as src, uncompressed.open('wb') as dst:
                        shutil.copyfileobj(src, dst)
                    saved.append(str(uncompressed.relative_to(UPLOAD_ROOT)))
                else:
                    saved.append(str(target.relative_to(UPLOAD_ROOT)))
            if 'invalidate_setup' in globals():
                invalidate_setup()
            refresh_card_files()
            listed = '<br>'.join(f'• <code>{html.escape(x)}</code>' for x in sorted(saved))
            display(HTML(f'<div class="mf-good"><b>✓ Files saved.</b><br>{listed}</div>'))
        except Exception as exc:
            display(HTML(f'<div class="mf-error"><b>Upload problem:</b><br>{html.escape(str(exc))}</div>'))
            return


def _module_help(module):
    return {
        'mod_I': 'Choose a TSV list containing species, accession, and/or taxonID columns.',
        'mod_II': 'Choose a TSV list containing accession or fetch_id.',
        'mod_III': 'Choose a protein FASTA. A metadata TSV is optional.',
    }[module]


def add_source_card(_button=None, initial_module='mod_III'):
    index = len(CUSTOM_SOURCE_CARDS) + 1
    enabled = widgets.Checkbox(value=True, description='Include this source', indent=False)
    label = widgets.Text(
        value=f'MY_SOURCE_{index}', description='Source label:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='470px')
    )
    group = widgets.Text(
        value='user_supplied', description='Source group:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='470px')
    )
    module = widgets.Dropdown(
        options=[('Module I — retrieve proteomes', 'mod_I'),
                 ('Module II — retrieve protein accessions', 'mod_II'),
                 ('Module III — use my FASTA', 'mod_III')],
        value=initial_module, description='Input type:',
        style={'description_width': 'initial'}, layout=widgets.Layout(width='570px')
    )
    primary_file = widgets.Dropdown(
        options=[('', '')] + [(f, f) for f in available_upload_files()],
        description='Main file:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='720px')
    )
    metadata_file = widgets.Dropdown(
        options=[('', '')] + [(f, f) for f in available_upload_files()],
        description='Metadata TSV:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='720px')
    )
    backend = widgets.Dropdown(
        options=['ncbi', 'uniprot', 'uniparc', 'pdb'], value='ncbi',
        description='Default fetch source:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px')
    )
    advanced_options = widgets.Text(
        value='', placeholder='Optional key=value;key=value (do not repeat fetch_source)',
        description='Advanced options:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='820px')
    )
    version = widgets.Text(value='', description='Version:', style={'description_width': 'initial'}, layout=widgets.Layout(width='410px'))
    citation = widgets.Text(value='', description='Citation/DOI:', style={'description_width': 'initial'}, layout=widgets.Layout(width='620px'))
    notes = widgets.Textarea(value='', description='Notes:', style={'description_width': 'initial'}, layout=widgets.Layout(width='820px', height='70px'))
    help_html = widgets.HTML()
    remove = widgets.Button(description='Remove card', icon='trash', button_style='warning', layout=widgets.Layout(width='150px'))

    card = {
        'enabled': enabled, 'label': label, 'group': group, 'module': module,
        'primary_file': primary_file, 'metadata_file': metadata_file,
        'backend': backend, 'advanced_options': advanced_options,
        'version': version, 'citation': citation, 'notes': notes,
    }

    optional_box = widgets.VBox([advanced_options, version, citation, notes])
    accordion = widgets.Accordion(children=[optional_box], selected_index=None)
    accordion.set_title(0, 'Optional provenance and advanced options')

    container = widgets.VBox(
        [widgets.HTML(f'<div class="mf-title">Custom source {index}</div>'), enabled,
         widgets.HBox([label, group]), module, help_html, primary_file,
         metadata_file, backend, accordion, remove],
        layout=widgets.Layout(border='1px solid #d7e6ee', padding='15px 17px', margin='0 0 12px 0', width='100%')
    )
    card['container'] = container
    CUSTOM_SOURCE_CARDS.append(card)
    if 'invalidate_setup' in globals():
        for watched in [enabled, label, group, module, primary_file, metadata_file,
                        backend, advanced_options, version, citation, notes]:
            watched.observe(invalidate_setup, names='value')
        invalidate_setup()

    def update_for_module(change=None):
        mod = module.value
        help_html.value = f'<div class="mf-muted" style="margin:2px 0 5px 105px">{_module_help(mod)}</div>'
        metadata_file.layout.display = '' if mod == 'mod_III' else 'none'
        backend.layout.display = 'none' if mod == 'mod_III' else ''
        if mod == 'mod_I':
            backend.options = ['ncbi', 'uniprot']
            if backend.value not in backend.options:
                backend.value = 'ncbi'
        elif mod == 'mod_II':
            backend.options = ['ncbi', 'uniprot', 'uniparc', 'pdb']
        primary_file.description = 'Protein FASTA:' if mod == 'mod_III' else 'Input list TSV:'

    def remove_card(_b=None):
        if card in CUSTOM_SOURCE_CARDS:
            CUSTOM_SOURCE_CARDS.remove(card)
        custom_cards_box.children = tuple(c['container'] for c in CUSTOM_SOURCE_CARDS)
        if 'invalidate_setup' in globals():
            invalidate_setup()

    module.observe(update_for_module, names='value')
    remove.on_click(remove_card)
    update_for_module()
    custom_cards_box.children = tuple(c['container'] for c in CUSTOM_SOURCE_CARDS)

save_uploads_button.on_click(save_uploads)
add_proteome_button.on_click(lambda _b: add_source_card(initial_module='mod_I'))
add_protein_button.on_click(lambda _b: add_source_card(initial_module='mod_II'))
add_fasta_button.on_click(lambda _b: add_source_card(initial_module='mod_III'))

template_download_output = widgets.Output()
template_download_buttons = []
for template_path in sorted(path for path in TEMPLATE_DIR.iterdir() if path.is_file()):
    button = widgets.Button(
        description=f'Download {template_path.name}',
        icon='download', button_style='info',
        layout=widgets.Layout(width='350px', height='38px'),
    )
    def _download_template(_button, path=template_path):
        with template_download_output:
            clear_output()
            try:
                from google.colab import files
                files.download(str(path))
            except Exception as exc:
                display(HTML(
                    '<div class="mf-error"><b>Download could not start:</b> '
                    f'{html.escape(str(exc))}</div>'
                ))
    button.on_click(_download_template)
    template_download_buttons.append(button)

display(HTML('<div class="mf-section"><div class="mf-title">1. Optional starter templates</div><div class="mf-muted">Download and edit one of these small examples only when useful.</div></div>'))
display(widgets.VBox(template_download_buttons), template_download_output)
display(HTML('<div class="mf-section"><div class="mf-title">2. Upload your files</div><div class="mf-muted">Choose all files for this run, then click Save uploaded files once.</div></div>'))
display(upload_widget, save_uploads_button, upload_output)
display(HTML('<div class="mf-section"><div class="mf-title">3. Tell maniFasta what each file is</div><div class="mf-muted">Click the button matching your main file. Repeat for additional sources.</div></div>'))
display(widgets.HBox([add_proteome_button, add_protein_button, add_fasta_button]), custom_cards_box)


<div style="background:#f0fdf4; border-left:7px solid #16a34a; border-radius:16px; padding:18px 20px; margin:18px 0;">
  <div style="font-size:23px; font-weight:900; color:#0f172a;">Question 4 — What should this run be called?</div>
  <p style="color:#334155; line-height:1.6; margin-bottom:0;">Choose a short run label and edit the database protein prefix. The prefix will begin every protein identifier in the final database FASTA (for example, <code>MyDB-00000000001</code>).</p>
</div>

<div style="background:#fefce8; border-left:7px solid #ca8a04; border-radius:14px; padding:14px 16px; margin-bottom:10px;">
  <b>Edit the database protein prefix.</b> The generic default <code>maniFastaDB</code> is valid, but a short project-specific prefix makes final FASTA identifiers easier to recognize and cite.
</div>

<div style="background:#eef6ff; border:1px solid #bfdbfe; border-radius:14px; padding:14px 16px; margin-bottom:8px;">
  <b>NCBI email and API key are optional.</b> They are most helpful for larger jobs that retrieve data from NCBI or perform taxonomy enrichment.<br>
  <span style="color:#475569;">The API key field is hidden. Credentials are written only to this temporary Colab workspace and are not placed in the final run ZIP.</span>
</div>

<div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:14px; padding:13px 16px; margin:10px 0;">
  <b>Need credentials?</b>
  <a href="https://www.ncbi.nlm.nih.gov/account/" target="_blank">NCBI account</a>
  &nbsp;·&nbsp;
  <a href="https://support.nlm.nih.gov/kbArticle/?pn=KA-05317" target="_blank">API-key instructions</a>
</div>

<div style="background:#fefce8; border-left:7px solid #ca8a04; border-radius:14px; padding:15px 18px; margin:12px 0;">
  <b>Checkpoint:</b> after any source, upload, run-label, database-prefix, taxonomy-output, or credential change, click <b>Validate & write setup</b> again before running.
</div>


In [ ]:
#@title Step 4. Name, review, and validate the run { display-mode: "form" }
#@markdown Enter a run label, edit the final FASTA protein prefix, and optionally add NCBI credentials.
#@markdown Click **Validate & write setup** after your final source and upload choices.

run_label_widget = widgets.Text(
    value='my_maniFasta_database',
    description='Run label:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='650px')
)
db_prefix_widget = widgets.Text(
    value='maniFastaDB',
    placeholder='Short prefix for every final FASTA protein ID',
    description='Database protein prefix:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='650px')
)
taxonomy_outputs_widget = widgets.Checkbox(
    value=True,
    description='Generate lineage table and taxonomy sunburst',
    indent=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='650px')
)
taxonomy_outputs_help = widgets.HTML(
    '<div class="mf-muted" style="margin:0 0 10px 0;max-width:820px">'
    'For very large databases, uncheck this to skip the memory-intensive lineage and plotting steps. '
    'The final FASTA and manifest are still produced.</div>'
)
ncbi_email_widget = widgets.Text(
    value='', placeholder='name@institution.edu',
    description='NCBI email:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='650px')
)
ncbi_key_widget = widgets.Password(
    value='', placeholder='Optional, but much faster for NCBI requests',
    description='NCBI API key:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='650px')
)
validate_button = widgets.Button(
    description='Validate & write setup', icon='check-circle', button_style='success',
    layout=widgets.Layout(width='245px', height='40px')
)
validation_output = widgets.Output()

REGISTRY_COLUMNS = [
    'source_id', 'enabled', 'source_label', 'source_group', 'module', 'collection',
    'source_dir', 'input_list', 'fasta_file', 'metadata_file', 'options',
    'version', 'citation', 'notes'
]


def validation_marker_path():
    return REPO_DIR / '00.setup' / '.maniFasta_colab_setup_validated'


def setup_digest(paths):
    h = hashlib.sha256()
    for path in paths:
        path = Path(path)
        h.update(str(path).encode('utf-8'))
        h.update(b'\0')
        h.update(path.read_bytes())
        h.update(b'\0')
    return h.hexdigest()


def invalidate_setup(change=None):
    validation_marker_path().unlink(missing_ok=True)


for watched in [run_label_widget, db_prefix_widget, taxonomy_outputs_widget, ncbi_email_widget, ncbi_key_widget,
                SOURCE_CONTROLS['human'], SOURCE_CONTROLS['homd']]:
    watched.observe(invalidate_setup, names='value')
for watched in SOURCE_CONTROLS['other'].values():
    watched.observe(invalidate_setup, names='value')
for card in CUSTOM_SOURCE_CARDS:
    for key in ('enabled', 'label', 'group', 'module', 'primary_file', 'metadata_file',
                'backend', 'advanced_options', 'version', 'citation', 'notes'):
        card[key].observe(invalidate_setup, names='value')


def sanitize_token(value: str, fallback='source'):
    token = re.sub(r'[^A-Za-z0-9_.-]+', '_', (value or '').strip()).strip('._-')
    return token or fallback


def selected_prepackaged_ids():
    ids = []
    if SOURCE_CONTROLS['human'].value:
        ids.append(SOURCE_CONTROLS['human'].value)
    if SOURCE_CONTROLS['homd'].value:
        ids.append(SOURCE_CONTROLS['homd'].value)
    ids += [sid for sid, cb in SOURCE_CONTROLS['other'].items() if cb.value]
    return ids


def parse_advanced_options(raw: str):
    opts = []
    for item in (raw or '').split(';'):
        item = item.strip()
        if not item:
            continue
        if '=' not in item:
            raise ValueError(f'Advanced option must be key=value: {item!r}')
        k, v = item.split('=', 1)
        if not k.strip():
            raise ValueError(f'Advanced option has an empty key: {item!r}')
        if k.strip().lower() == 'fetch_source':
            raise ValueError('Choose fetch_source from the menu; do not repeat it in Advanced options.')
        opts.append(f'{k.strip()}={v.strip()}')
    return opts


def custom_registry_rows():
    rows = []
    for card in CUSTOM_SOURCE_CARDS:
        if not card['enabled'].value:
            continue
        label = sanitize_token(card['label'].value, 'USER_SOURCE')
        group = sanitize_token(card['group'].value, 'user_supplied')
        module = card['module'].value
        primary = (card['primary_file'].value or '').strip()
        metadata = (card['metadata_file'].value or '').strip()
        if not primary:
            raise ValueError(f'{label}: choose a main input file.')
        main_path = UPLOAD_ROOT / primary
        if not main_path.is_file():
            raise FileNotFoundError(f'{label}: uploaded file not found: {primary}')
        if metadata and not (UPLOAD_ROOT / metadata).is_file():
            raise FileNotFoundError(f'{label}: metadata file not found: {metadata}')

        opts = []
        if module in {'mod_I', 'mod_II'}:
            opts.append(f'fetch_source={card["backend"].value}')
        opts.extend(parse_advanced_options(card['advanced_options'].value))

        row = {c: '' for c in REGISTRY_COLUMNS}
        row.update({
            'source_id': 'USER_' + sanitize_token(label).upper(),
            'enabled': 'TRUE',
            'source_label': label,
            'source_group': group,
            'module': module,
            'collection': '',
            'source_dir': '00.setup/user_uploads',
            'options': ';'.join(opts),
            'version': card['version'].value.strip(),
            'citation': card['citation'].value.strip(),
            'notes': card['notes'].value.strip(),
        })
        if module in {'mod_I', 'mod_II'}:
            row['input_list'] = primary
        else:
            row['fasta_file'] = primary
            row['metadata_file'] = metadata
        rows.append(row)
    return rows


def set_config_assignment(text: str, key: str, value: str):
    quoted = "'" + value.replace("'", "'\\''") + "'"
    pattern = re.compile(rf'(?m)^{re.escape(key)}=.*$')
    line = f'{key}={quoted}'
    return pattern.sub(line, text, count=1) if pattern.search(text) else text.rstrip() + '\n' + line + '\n'


def write_registry_and_config(_button=None):
    phase_started = datetime.now(timezone.utc)
    phase_status = 'failed'
    phase_detail = ''
    api_key_value = ncbi_key_widget.value.strip()
    with validation_output:
        clear_output()
        try:
            invalidate_setup()
            if not SOURCE_ROWS:
                raise RuntimeError('Run Step 2 first to load the repository registry.')

            label = sanitize_token(run_label_widget.value, 'maniFasta_run')
            db_prefix = sanitize_token(db_prefix_widget.value, 'maniFastaDB')
            taxonomy_outputs = 'true' if taxonomy_outputs_widget.value else 'false'
            selected = set(selected_prepackaged_ids())
            custom = custom_registry_rows()

            rows = []
            for original in SOURCE_ROWS:
                row = {c: (original.get(c) or '') for c in REGISTRY_COLUMNS}
                row['enabled'] = 'TRUE' if row['source_id'] in selected else 'FALSE'
                rows.append(row)
            rows.extend(custom)

            enabled_rows = [r for r in rows if truthy(r['enabled'])]
            if not enabled_rows:
                raise ValueError('Select at least one prepackaged or custom source.')

            human_enabled = [r for r in enabled_rows if r['collection'].strip().lower() == 'human_uniprot']
            homd_enabled = [r for r in enabled_rows if r['collection'].strip().lower() == 'homd']
            if len(human_enabled) > 1:
                raise ValueError('Choose no more than one human protein set.')
            if len(homd_enabled) > 1:
                raise ValueError('Choose no more than one HOMD flavor.')

            source_ids = [r['source_id'] for r in rows]
            if len(source_ids) != len(set(source_ids)):
                dup = sorted({x for x in source_ids if source_ids.count(x) > 1})
                raise ValueError(f'Duplicate source_id values: {dup}')
            labels = [r['source_label'] for r in enabled_rows]
            if len(labels) != len(set(labels)):
                dup = sorted({x for x in labels if labels.count(x) > 1})
                raise ValueError(f'Enabled source labels must be unique: {dup}')

            registry = REPO_DIR / '00.setup' / '000.maniFasta.source_registry.tsv'
            backup = REPO_DIR / '00.setup' / '000.maniFasta.source_registry.repo_default.tsv'
            if registry.is_file() and not backup.exists():
                shutil.copy2(registry, backup)
            with registry.open('w', encoding='utf-8', newline='') as fh:
                writer = csv.DictWriter(fh, fieldnames=REGISTRY_COLUMNS, delimiter='\t', lineterminator='\n')
                writer.writeheader()
                writer.writerows(rows)

            config = REPO_DIR / '00.setup' / '000.maniFasta.config'
            text = config.read_text(encoding='utf-8')
            for key, value in {
                'mainDIR': str(REPO_DIR),
                'dbRootDIR': str(RUN_ROOT),
                'logDIR': str(RUN_ROOT),
                'run_label': label,
                'db_prefix': db_prefix,
                'add_lineage': taxonomy_outputs,
                'plot_taxonomy': taxonomy_outputs,
                'taxonomy_plot_png': 'false',
                'write_db_summary': 'true',
                'lineage_allow_fetch': 'true',
            }.items():
                text = set_config_assignment(text, key, value)
            config.write_text(text, encoding='utf-8')

            datasets_path = shutil.which('datasets') or '/usr/local/bin/datasets'
            local_env = REPO_DIR / '00.setup' / '000.maniFasta.local.env'
            local_env.write_text(
                f'export DATASETS_BIN={datasets_path!r}\n'
                f'export NCBI_EMAIL={ncbi_email_widget.value.strip()!r}\n'
                f'export NCBI_API_KEY={api_key_value!r}\n',
                encoding='utf-8'
            )
            local_env.chmod(0o600)
            # Do not retain the key in notebook widget state. The run-scoped,
            # mode-0600 environment file remains available to the build.
            ncbi_key_widget.value = ''
            api_key_value = ''

            check_dir = CONTENT_DIR / '.manifasta_plan_check'
            if check_dir.exists():
                shutil.rmtree(check_dir)
            check_dir.mkdir(parents=True)
            cmd = [
                sys.executable, str(REPO_DIR / '01.scripts' / 'source_planner.py'),
                '--registry', str(registry),
                '--main-dir', str(REPO_DIR),
                '--setup-dir', str(REPO_DIR / '00.setup'),
                '--ref-dir', str(check_dir / 'run_data'),
                '--tmp-dir', str(check_dir / 'tmp'),
                '--out-normalized-plan', str(check_dir / 'source_plan.normalized.tsv'),
            ]
            result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            if result.returncode != 0:
                raise RuntimeError('source_planner.py rejected the generated registry:\n' + result.stdout)

            digest = setup_digest([registry, config, local_env])
            validation_marker_path().write_text(digest + '\n', encoding='utf-8')

            rows_html = ''.join(
                '<tr>'
                f'<td><code>{html.escape(r["source_label"])}</code></td>'
                f'<td>{html.escape(r["source_group"] or "—")}</td>'
                f'<td>{html.escape(r["module"])}</td>'
                f'<td>{html.escape(r["collection"] or r["input_list"] or r["fasta_file"] or "—")}</td>'
                '</tr>'
                for r in enabled_rows
            )
            display(HTML(
                f'<div class="mf-good"><b>✓ Setup validated.</b> {len(enabled_rows)} source(s) will be included.<br>'
                f'Run label: <code>{label}</code><br>'
                f'Final FASTA protein prefix: <code>{db_prefix}</code><br>'
                f'Taxonomy outputs: <code>{taxonomy_outputs}</code></div>'
                '<div style="overflow-x:auto;margin-top:10px"><table style="border-collapse:collapse;width:100%">'
                '<tr style="background:#eef5f8"><th style="padding:7px;text-align:left">Source</th>'
                '<th style="padding:7px;text-align:left">Group</th><th style="padding:7px;text-align:left">Module</th>'
                '<th style="padding:7px;text-align:left">Input / collection</th></tr>' + rows_html + '</table></div>'
            ))
            phase_status = 'completed'
            if db_prefix == 'maniFastaDB':
                display(HTML('<div class="mf-note" style="margin-top:10px"><b>Generic prefix retained.</b> This is valid, but consider a project-specific prefix before the production build.</div>'))
            if not ncbi_email_widget.value.strip():
                display(HTML('<div class="mf-note" style="margin-top:10px"><b>NCBI email is blank.</b> The build may still work, but an email and optional API key are recommended for reliable NCBI requests and faster taxonomy enrichment.</div>'))
        except Exception as exc:
            phase_detail = str(exc)
            display(HTML(f'<div class="mf-error"><b>Setup validation failed:</b><br><pre style="white-space:pre-wrap">{html.escape(str(exc))}</pre></div>'))
        finally:
            # Clear after both success and failure. This is essential because
            # Jupyter can otherwise persist Password widget values in metadata.
            ncbi_key_widget.value = ''
            api_key_value = ''
            _record_colab_phase('validate_and_write_setup', phase_started, status=phase_status, detail=phase_detail)

validate_button.on_click(write_registry_and_config)

display(HTML('<div class="mf-section"><div class="mf-title">Run identity and NCBI access</div><div class="mf-muted">The API key is hidden on screen and written only to this temporary Colab runtime. It is not copied into the final run package by maniFasta.</div></div>'))
display(run_label_widget, db_prefix_widget, taxonomy_outputs_widget, taxonomy_outputs_help, ncbi_email_widget, ncbi_key_widget, validate_button, validation_output)


<div style="background:#fefce8; border-left:7px solid #ca8a04; border-radius:16px; padding:18px 20px; margin:18px 0;">
  <div style="font-size:23px; font-weight:900; color:#0f172a;">Step 5 — Run the build</div>
  <p style="color:#334155; line-height:1.6; margin-bottom:0;">Run this cell after the validation checkpoint is green. Terminal output remains visible throughout the build, and a heartbeat appears only when maniFasta has been quiet.</p>
</div>

<div style="background:#eef6ff; border:1px solid #bfdbfe; border-radius:14px; padding:14px 16px; margin:12px 0;">
  <b>Run records:</b> terminal output is saved under <code>run_logs/</code>. Step 5 also creates <code>COLAB_RUNTIME_INFO.tsv</code> and <code>COLAB_PHASE_TIMINGS.tsv</code> with elapsed wall time, CPU, RAM, accelerator, operating-system, Python, disk, and maniFasta Git information. Both are included in the complete results ZIP.
</div>

<div style="background:#fff7ed; border:1px solid #fed7aa; border-radius:14px; padding:14px 16px; margin:12px 0;">
  <b>Colab limitation:</b> if the runtime is deleted, restarted, or stopped because it runs out of memory or disk space, files under <code>/content</code> are removed. Large HOMD, human TrEMBL, pan-proteome, or extensive accession builds may exceed free-Colab limits.
</div>


In [ ]:
#@title Step 5. Run maniFasta and package the complete run { display-mode: "form" }
#@markdown Keep this cell open while it runs. Terminal output streams continuously, and a heartbeat appears only during quiet periods.
#@markdown The run ZIP includes the full log and a Methods-ready COLAB_RUNTIME_INFO.tsv resource report.

RUN_MANIFASTA_NOW = True #@param {type:"boolean"}
HEARTBEAT_SECONDS = 60 #@param {type:"integer"}

from datetime import datetime, timezone
from IPython.display import SVG, Markdown
import codecs
import os
import platform
import selectors
import subprocess
import time

def newest_run_dir():
    dirs = [p for p in RUN_ROOT.glob('*') if p.is_dir()]
    return max(dirs, key=lambda p: p.stat().st_mtime) if dirs else None

def package_complete_run(run_dir: Path) -> Path:
    zip_base = CONTENT_DIR / run_dir.name
    return Path(shutil.make_archive(
        str(zip_base), 'zip', root_dir=run_dir.parent, base_dir=run_dir.name
    ))

def _command_output(command):
    result = subprocess.run(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    return result.stdout.strip()

def _read_meminfo():
    values = {}
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            key, value = line.split(':', 1)
            values[key] = int(value.strip().split()[0]) * 1024
    except Exception:
        pass
    return values

def _runtime_details():
    disk = shutil.disk_usage(CONTENT_DIR)
    meminfo = _read_meminfo()

    cpu_model = _command_output(
        "lscpu | sed -n 's/^Model name:[[:space:]]*//p' | head -1"
    )
    if not cpu_model:
        cpu_model = _command_output(
            "sed -n 's/^model name[[:space:]]*:[[:space:]]*//p' /proc/cpuinfo | head -1"
        )

    gpu = _command_output(
        "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null"
    )
    tpu = os.environ.get('TPU_NAME') or os.environ.get('COLAB_TPU_ADDR')
    if gpu:
        runtime_type = 'GPU'
        accelerator = '; '.join(line.strip() for line in gpu.splitlines() if line.strip())
    elif tpu:
        runtime_type = 'TPU'
        accelerator = str(tpu)
    else:
        runtime_type = 'CPU'
        accelerator = 'None'

    return {
        'cpu_model': cpu_model or 'Unavailable',
        'logical_cpus_available': os.cpu_count() or '',
        'total_ram_gb': meminfo.get('MemTotal', 0) / (1024 ** 3) if meminfo else '',
        'available_ram_gb': meminfo.get('MemAvailable', 0) / (1024 ** 3) if meminfo else '',
        'runtime_type': runtime_type,
        'accelerator': accelerator,
        'operating_system': platform.platform(),
        'python_version': platform.python_version(),
        'disk_total_gb': disk.total / (1024 ** 3),
        'disk_free_gb': disk.free / (1024 ** 3),
        'colab_release_tag': os.environ.get('COLAB_RELEASE_TAG', ''),
        'colab_backend_version': os.environ.get('COLAB_BACKEND_VERSION', ''),
    }

def _format_value(value):
    if isinstance(value, float):
        return f'{value:.3f}'
    return str(value)

def _runtime_snapshot(details=None):
    details = details or _runtime_details()
    disk_text = f"{details['disk_free_gb']:.1f} GB disk free"
    ram_value = details.get('available_ram_gb', '')
    ram_text = (
        f'{ram_value:.1f} GB RAM available'
        if isinstance(ram_value, (int, float))
        else 'RAM unavailable'
    )
    return f'{disk_text}; {ram_text}'

def _write_colab_runtime_info(
    run_dir: Path,
    start_time,
    finish_time,
    return_code,
    start_details,
    finish_details,
):
    elapsed_seconds = max(0.0, (finish_time - start_time).total_seconds())
    git_commit = _command_output(f'git -C "{REPO_DIR}" rev-parse HEAD')
    git_status = _command_output(f'git -C "{REPO_DIR}" status --porcelain')
    git_branch = _command_output(f'git -C "{REPO_DIR}" branch --show-current')

    rows = [
        ('report_file', 'COLAB_RUNTIME_INFO.tsv'),
        ('timing_definition', 'Wall-clock time from immediately before launching bash runBuild_db.sh until process exit; final ZIP packaging is excluded.'),
        ('started_utc', start_time.isoformat().replace('+00:00', 'Z')),
        ('finished_utc', finish_time.isoformat().replace('+00:00', 'Z')),
        ('elapsed_seconds', elapsed_seconds),
        ('elapsed_minutes', elapsed_seconds / 60),
        ('exit_code', return_code),
        ('run_directory', run_dir.name),
        ('runtime_type', start_details.get('runtime_type', '')),
        ('accelerator', start_details.get('accelerator', '')),
        ('cpu_model', start_details.get('cpu_model', '')),
        ('logical_cpus_available', start_details.get('logical_cpus_available', '')),
        ('total_ram_gb', start_details.get('total_ram_gb', '')),
        ('available_ram_at_start_gb', start_details.get('available_ram_gb', '')),
        ('available_ram_at_finish_gb', finish_details.get('available_ram_gb', '')),
        ('disk_total_gb', start_details.get('disk_total_gb', '')),
        ('disk_free_at_start_gb', start_details.get('disk_free_gb', '')),
        ('disk_free_at_finish_gb', finish_details.get('disk_free_gb', '')),
        ('operating_system', start_details.get('operating_system', '')),
        ('python_version', start_details.get('python_version', '')),
        ('colab_release_tag', start_details.get('colab_release_tag', '')),
        ('colab_backend_version', start_details.get('colab_backend_version', '')),
        ('manifasta_git_commit', git_commit or 'Unavailable'),
        ('manifasta_git_branch', git_branch or 'Unavailable'),
        ('manifasta_working_tree', 'modified' if git_status else 'clean'),
    ]

    report_path = run_dir / 'COLAB_RUNTIME_INFO.tsv'
    report_path.write_text(
        'field\tvalue\n' +
        ''.join(
            f'{field}\t{_format_value(value).replace(chr(9), " ").replace(chr(10), "; ")}\n'
            for field, value in rows
        ),
        encoding='utf-8',
    )
    return report_path, elapsed_seconds

def _show_run_results(run_dir: Path, zip_path: Path, return_code: int):
    status_class = 'mf-good' if return_code == 0 else 'mf-error'
    status_text = 'completed successfully' if return_code == 0 else f'exited with status {return_code}'
    display(HTML(
        f'<div class="{status_class}"><b>maniFasta {status_text}.</b><br>'
        f'Run directory: <code>{run_dir}</code><br>'
        f'Complete ZIP: <code>{zip_path.name}</code></div>'
    ))

    summaries = sorted(run_dir.glob('*.database_methods.md'))
    if summaries:
        display(Markdown('### Database methods summary'))
        text = summaries[0].read_text(encoding='utf-8', errors='replace')
        display(Markdown(text[:12000] + ('\n\n…' if len(text) > 12000 else '')))

    svgs = sorted(run_dir.glob('*.taxonomy_sunburst.svg'))
    html_sunbursts = sorted(run_dir.glob('*.taxonomy_sunburst.html'))
    if svgs:
        display(Markdown('### Taxonomy sunburst preview'))
        display(SVG(filename=str(svgs[0])))
    else:
        display(HTML(
            '<div class="mf-note"><b>No sunburst preview was found.</b> '
            'Check the run log and lineage outputs; the main FASTA build may still have succeeded.</div>'
        ))

    if html_sunbursts:
        interactive_path = html_sunbursts[0]
        display(HTML(
            '<div class="mf-good" style="margin-top:14px">'
            '<b>Interactive taxonomy sunburst</b><br>'
            f'<code>{html.escape(interactive_path.name)}</code><br>'
            'Download this self-contained HTML file, then open it in your web browser. '
            'It is also included in the complete run ZIP.</div>'
        ))
        download_sunburst_button = widgets.Button(
            description='Download interactive HTML',
            icon='download',
            button_style='info',
            layout=widgets.Layout(width='265px', height='38px'),
        )
        download_sunburst_output = widgets.Output()
        def _download_interactive_sunburst(_button):
            with download_sunburst_output:
                clear_output()
                try:
                    from google.colab import files
                    files.download(str(interactive_path))
                except Exception as exc:
                    display(HTML(
                        '<div class="mf-error"><b>Download could not start:</b> '
                        f'{html.escape(str(exc))}</div>'
                    ))
        download_sunburst_button.on_click(_download_interactive_sunburst)
        display(download_sunburst_button, download_sunburst_output)
    elif svgs:
        display(HTML(
            '<div class="mf-note"><b>Static preview only.</b> '
            'No interactive taxonomy HTML was generated for this run.</div>'
        ))

    key_files = []
    for pattern in (
        '*.fasta', '*.manifest.tsv', '*.manifest.lineage.tsv',
        'build_summary.tsv', '*.database_methods.md',
        '*.taxonomy_sunburst.svg', '*.taxonomy_sunburst.html',
        'COLAB_RUNTIME_INFO.tsv', 'COLAB_PHASE_TIMINGS.tsv',
        'COLAB_RUN_STATUS.txt', 'run_logs/*.log'
    ):
        key_files.extend(run_dir.glob(pattern))
    if key_files:
        display(Markdown('### Key outputs'))
        output_items = ''.join(
            f'<li><code>{html.escape(str(path.relative_to(run_dir)))}</code></li>'
            for path in sorted(set(key_files))
        )
        display(HTML(
            '<div class="mf-section"><div class="mf-muted">These files are included in the '
            'complete run ZIP downloaded in Step 6.</div><ul>' + output_items + '</ul></div>'
        ))

def _make_pre_run_failure_dir(start_stamp: str, fallback_log: Path) -> Path:
    failed = RUN_ROOT / f'{start_stamp}_FAILED_BEFORE_RUN_DIRECTORY'
    failed.mkdir(parents=True, exist_ok=True)
    (failed / 'run_logs').mkdir(exist_ok=True)
    shutil.copy2(fallback_log, failed / 'run_logs' / fallback_log.name)
    for source in (
        REPO_DIR / '00.setup' / '000.maniFasta.source_registry.tsv',
        REPO_DIR / '00.setup' / '000.maniFasta.config',
    ):
        if source.is_file():
            shutil.copy2(source, failed / source.name)
    return failed

if not RUN_MANIFASTA_NOW:
    display(HTML(
        '<div class="mf-note"><b>The build has not started.</b> '
        'Check RUN_MANIFASTA_NOW and run this cell again when ready.</div>'
    ))
else:
    registry = REPO_DIR / '00.setup' / '000.maniFasta.source_registry.tsv'
    config = REPO_DIR / '00.setup' / '000.maniFasta.config'
    local_env = REPO_DIR / '00.setup' / '000.maniFasta.local.env'
    marker = REPO_DIR / '00.setup' / '.maniFasta_colab_setup_validated'

    if not registry.is_file() or not config.is_file() or not local_env.is_file() or not marker.is_file():
        raise RuntimeError('Complete Step 4 and click “Validate & write setup” after your final choices.')
    current_digest = setup_digest([registry, config, local_env])
    if marker.read_text(encoding='utf-8').strip() != current_digest:
        raise RuntimeError('The setup changed after validation. Return to Step 4 and validate it again.')

    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    LIVE_LOG_DIR = CONTENT_DIR / 'maniFasta_live_logs'
    LIVE_LOG_DIR.mkdir(parents=True, exist_ok=True)

    before = {p.resolve() for p in RUN_ROOT.glob('*') if p.is_dir()}
    start_time = datetime.now(timezone.utc)
    start_stamp = start_time.strftime('%Y%m%d_%H%M%S')
    fallback_log = LIVE_LOG_DIR / f'{start_stamp}.manifasta_colab_live.log'
    runtime_start = _runtime_details()

    env = os.environ.copy()
    env['Config_file'] = str(config)
    env['PYTHONUNBUFFERED'] = '1'

    print('Starting maniFasta…', flush=True)
    print(f'Live log: {fallback_log}', flush=True)
    print(f'Runtime resources: {_runtime_snapshot(runtime_start)}', flush=True)
    print('The next heartbeat will appear even if a download or database step is quiet.', flush=True)
    print('=' * 78, flush=True)

    cmd = ['stdbuf', '-oL', '-eL', 'bash', 'runBuild_db.sh']
    active_run_dir = None
    mirror_log_path = None
    mirror_handle = None
    return_code = 1

    with fallback_log.open('w', encoding='utf-8') as live_log:
        live_log.write('[maniFasta Colab live log]\n')
        live_log.write(f'started_utc={start_time.isoformat()}\n')
        live_log.write(f'cwd={REPO_DIR / "01.scripts"}\n')
        live_log.write(f'Config_file={config}\n')
        live_log.write(f'resources_at_start={_runtime_snapshot(runtime_start)}\n')
        live_log.write('=' * 78 + '\n')
        live_log.flush()

        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_DIR / '01.scripts'),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=0,
        )
        assert proc.stdout is not None
        pipe_fd = proc.stdout.fileno()
        os.set_blocking(pipe_fd, False)
        selector = selectors.DefaultSelector()
        selector.register(pipe_fd, selectors.EVENT_READ)
        decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')

        last_activity = time.monotonic()
        last_run_check = 0.0
        pipe_open = True

        def emit(text):
            print(text, end='', flush=True)
            live_log.write(text)
            live_log.flush()
            if mirror_handle is not None:
                mirror_handle.write(text)
                mirror_handle.flush()

        try:
            while pipe_open or proc.poll() is None:
                now = time.monotonic()

                if now - last_run_check >= 3:
                    last_run_check = now
                    new_dirs = [
                        p for p in RUN_ROOT.glob('*')
                        if p.is_dir() and p.resolve() not in before
                    ]
                    if new_dirs and active_run_dir is None:
                        active_run_dir = max(new_dirs, key=lambda p: p.stat().st_mtime)
                        log_dir = active_run_dir / 'run_logs'
                        log_dir.mkdir(parents=True, exist_ok=True)
                        mirror_log_path = log_dir / f'{start_stamp}.{active_run_dir.name}.manifasta_colab_run.log'
                        shutil.copy2(fallback_log, mirror_log_path)
                        mirror_handle = mirror_log_path.open('a', encoding='utf-8')
                        emit(f'\n[COLAB] Run directory: {active_run_dir}\n')
                        emit(f'[COLAB] Run log: {mirror_log_path}\n')

                for key, _ in selector.select(timeout=1.0):
                    try:
                        chunk = os.read(key.fd, 65536)
                    except BlockingIOError:
                        continue
                    if chunk:
                        text = decoder.decode(chunk)
                        if text:
                            emit(text)
                        last_activity = time.monotonic()
                    else:
                        selector.unregister(key.fd)
                        pipe_open = False

                now = time.monotonic()
                if now - last_activity >= max(15, int(HEARTBEAT_SECONDS)):
                    elapsed = datetime.now(timezone.utc) - start_time
                    emit(
                        f'\n[COLAB HEARTBEAT] Still running after {str(elapsed).split(".")[0]} '
                        f'| {_runtime_snapshot()}\n'
                    )
                    last_activity = now

            tail = decoder.decode(b'', final=True)
            if tail:
                emit(tail)
            return_code = proc.wait()
        finally:
            selector.close()
            if mirror_handle is not None:
                mirror_handle.close()

        finish_time = datetime.now(timezone.utc)
        runtime_finish = _runtime_details()
        _record_colab_phase(
            'run_manifasta', start_time, finish_time,
            status='completed' if return_code == 0 else 'failed',
            detail=f'exit_code={return_code}',
        )
        live_log.write('=' * 78 + '\n')
        live_log.write(f'finished_utc={finish_time.isoformat()}\n')
        live_log.write(f'exit_code={return_code}\n')
        live_log.write(f'resources_at_finish={_runtime_snapshot(runtime_finish)}\n')
        live_log.flush()

    if active_run_dir is None:
        new_dirs = [
            p for p in RUN_ROOT.glob('*')
            if p.is_dir() and p.resolve() not in before
        ]
        active_run_dir = max(new_dirs, key=lambda p: p.stat().st_mtime) if new_dirs else None

    if active_run_dir is None:
        active_run_dir = _make_pre_run_failure_dir(start_stamp, fallback_log)
    else:
        log_dir = active_run_dir / 'run_logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        final_log = log_dir / f'{start_stamp}.{active_run_dir.name}.manifasta_colab_run.log'
        shutil.copy2(fallback_log, final_log)

    runtime_info_path, elapsed_seconds = _write_colab_runtime_info(
        active_run_dir,
        start_time,
        finish_time,
        return_code,
        runtime_start,
        runtime_finish,
    )

    status_file = active_run_dir / 'COLAB_RUN_STATUS.txt'
    status_file.write_text(
        f'started_utc={start_time.isoformat()}\n'
        f'finished_utc={finish_time.isoformat()}\n'
        f'elapsed_seconds={elapsed_seconds:.3f}\n'
        f'elapsed_minutes={elapsed_seconds / 60:.3f}\n'
        f'exit_code={return_code}\n'
        f'live_log={fallback_log}\n'
        f'runtime_report={runtime_info_path}\n'
        f'final_resources={_runtime_snapshot(runtime_finish)}\n',
        encoding='utf-8',
    )

    phase_timings_path = _write_colab_phase_timings(active_run_dir)
    zip_path = package_complete_run(active_run_dir)
    LAST_RUN_DIR, LAST_RUN_ZIP = active_run_dir, zip_path

    print('=' * 78, flush=True)
    print(f'Build process exited with status {return_code}.', flush=True)
    print(f'Elapsed build time: {elapsed_seconds / 60:.2f} minutes.', flush=True)
    print(f'Colab runtime report: {runtime_info_path}', flush=True)
    print(f'Colab phase timings: {phase_timings_path}', flush=True)
    print(f'Complete run package: {zip_path}', flush=True)
    _show_run_results(active_run_dir, zip_path, return_code)

    if return_code != 0:
        raise RuntimeError(
            f'maniFasta exited with status {return_code}. '
            f'The partial/failed run and live log were packaged in {zip_path.name}.'
        )


<div style="background:#f0fdf4; border-left:7px solid #16a34a; border-radius:16px; padding:18px 20px; margin:18px 0;">
  <div style="font-size:23px; font-weight:900; color:#0f172a;">Step 6 — Download your complete run</div>
  <p style="color:#334155; line-height:1.6; margin-bottom:0;">After Step 5 finishes, run the small download cell below. It finds the newest complete run ZIP even if the notebook display was refreshed.</p>
</div>

<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr)); gap:12px; margin:12px 0;">
  <div style="background:white; border:1px solid #bbf7d0; border-radius:14px; padding:14px;"><b>📄 FASTA + manifests</b><br>Final database and provenance files</div>
  <div style="background:white; border:1px solid #bbf7d0; border-radius:14px; padding:14px;"><b>🧾 Run log</b><br>Full Colab terminal output and heartbeat status</div>
  <div style="background:white; border:1px solid #bbf7d0; border-radius:14px; padding:14px;"><b>🧬 Taxonomy plots</b><br>Static SVG, interactive HTML, and lineage table</div>
  <div style="background:white; border:1px solid #bbf7d0; border-radius:14px; padding:14px;"><b>🔁 Reproducibility</b><br>Registry, config, and methods summary</div>
</div>

> **Download the ZIP before the Colab runtime is deleted.** Nothing is automatically saved to Google Drive.

In [ ]:
#@title Step 6. Download the newest complete run ZIP { display-mode: "form" }
DOWNLOAD_NEWEST_RUN_ZIP = True #@param {type:"boolean"}

from pathlib import Path
from IPython.display import display, HTML

CONTENT_DIR = Path('/content')
zip_candidates = sorted(
    CONTENT_DIR.glob('20??????_??????_*.zip'),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
zip_path = None
if 'LAST_RUN_ZIP' in globals() and LAST_RUN_ZIP and Path(LAST_RUN_ZIP).is_file():
    zip_path = Path(LAST_RUN_ZIP)
elif zip_candidates:
    zip_path = zip_candidates[0]

if zip_path is None:
    display(HTML(
        '<div class="mf-error"><b>No run ZIP was found.</b> '
        'Step 5 may not have reached the packaging stage, or the Colab runtime may have restarted.</div>'
    ))
else:
    display(HTML(
        f'<div class="mf-good"><b>Newest complete run package:</b><br>'
        f'<code>{zip_path}</code><br>Size: {zip_path.stat().st_size / (1024**2):.1f} MB</div>'
    ))
    if DOWNLOAD_NEWEST_RUN_ZIP:
        display(HTML(
            '<div class="mf-note"><b>Starting browser download…</b> Large ZIP files may take a moment to begin.</div>'
        ))
        from google.colab import files
        files.download(str(zip_path))
    else:
        display(HTML(
            '<div class="mf-note">Set <code>DOWNLOAD_NEWEST_RUN_ZIP</code> to true and rerun this cell to download the package.</div>'
        ))


<div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:18px; padding:20px 22px; margin:18px 0;">
  <div style="font-size:23px; font-weight:900; color:#0f172a; margin-bottom:10px;">Between production runs: use this order</div>

  <div style="background:#fee2e2; border-left:6px solid #dc2626; border-radius:12px; padding:14px; margin-bottom:12px;">
    <b>First: download and verify the complete results ZIP.</b> Everything under <code>/content</code> is temporary.
  </div>

  <ol style="color:#334155; line-height:1.75; padding-left:24px; margin-bottom:14px;">
    <li><b>Clear the maniFasta workspace:</b> run the workspace-clear cell below and confirm deletion.</li>
    <li><b>Clear notebook outputs:</b> choose <b>Edit → Clear all outputs</b>.</li>
    <li><b>Delete the old runtime:</b> choose <b>Runtime → Disconnect and delete runtime</b>.</li>
    <li><b>Start the next run:</b> return to Step 1 and press play; Colab will assign a new runtime.</li>
  </ol>

  <div style="background:#eef6ff; border-left:6px solid #2563eb; border-radius:12px; padding:14px; margin-bottom:10px;">
    <b>Why this order?</b> Running the workspace-clear cell creates a final confirmation output, so clear all outputs <i>after</i> workspace cleanup. Deleting the runtime then ensures that the next production build starts on a newly assigned Colab machine.
  </div>

  <div style="background:#f0fdf4; border-left:6px solid #16a34a; border-radius:12px; padding:14px;">
    <b>Keep the ZIP.</b> It contains the log, registry, config, manifests, outputs, and <code>COLAB_RUNTIME_INFO.tsv</code> and <code>COLAB_PHASE_TIMINGS.tsv</code> needed for reporting or troubleshooting.
  </div>
</div>


In [ ]:
#@title Optional. Clear this Colab workspace { display-mode: "form" }
#@markdown Use this only when you want to delete the cloned repo, uploads, run folders, live logs, and maniFasta run ZIPs from this Colab runtime.

from pathlib import Path
from IPython.display import display, HTML, clear_output
import html
import shutil
import subprocess
import sys

try:
    import ipywidgets as widgets
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=True)
    import ipywidgets as widgets

CONTENT_DIR = Path('/content')
REPO_DIR = CONTENT_DIR / 'maniFasta'
RUN_ROOT = CONTENT_DIR / 'maniFasta_runs'
TEMPLATE_DIR = CONTENT_DIR / 'maniFasta_input_templates'

confirm_reset = widgets.Checkbox(
    value=False,
    description='I understand that temporary maniFasta files in this runtime will be deleted.',
    indent=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='760px')
)
reset_button = widgets.Button(
    description='Clear this Colab workspace',
    icon='trash',
    button_style='warning',
    layout=widgets.Layout(width='270px')
)
reset_output = widgets.Output()


def _remove_path(path: Path):
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()


def reset_workspace(_button=None):
    global SOURCE_ROWS, SOURCE_CONTROLS, CUSTOM_SOURCE_CARDS, LAST_RUN_DIR, LAST_RUN_ZIP, MANIFASTA_GIT_COMMIT
    with reset_output:
        clear_output()
        if not confirm_reset.value:
            display(HTML(
                '<div style="background:#fff7e7;border:1px solid #efdcad;border-radius:12px;'
                'padding:12px 15px;color:#66582f"><b>Nothing was deleted.</b> '
                'Check the confirmation box first.</div>'
            ))
            return

        reset_button.disabled = True
        try:
            for path in [
                REPO_DIR,
                RUN_ROOT,
                TEMPLATE_DIR,
                CONTENT_DIR / 'maniFasta_live_logs',
                CONTENT_DIR / '.manifasta_plan_check',
            ]:
                _remove_path(path)

            for pattern in ('*.manifasta_colab_run.log', '20??????_??????_*.zip'):
                for path in CONTENT_DIR.glob(pattern):
                    path.unlink(missing_ok=True)

            SOURCE_ROWS = []
            SOURCE_CONTROLS = {}
            CUSTOM_SOURCE_CARDS = []
            LAST_RUN_DIR = None
            LAST_RUN_ZIP = None
            MANIFASTA_GIT_COMMIT = None
            confirm_reset.value = False

            display(HTML(
                '<div style="background:#edf8f0;border:1px solid #c5e4cc;border-radius:12px;'
                'padding:12px 15px;color:#315b3b"><b>✓ maniFasta workspace cleared.</b> '
                'Return to Question 1 and prepare the repository again.</div>'
            ))
        except Exception as exc:
            display(HTML(
                '<div style="background:#fff0f0;border:1px solid #efcaca;border-radius:12px;'
                f'padding:12px 15px;color:#7c3535"><b>Workspace cleanup failed:</b><br>{html.escape(str(exc))}</div>'
            ))
        finally:
            reset_button.disabled = False


reset_button.on_click(reset_workspace)
display(confirm_reset, reset_button, reset_output)
